# Google Shopping bias audit — Learning-to-rank (reduced dataset)

**This notebook is the *reduced* study**: it runs on the FIRST scraping dataset (`results.db`,
~53k rows, IT/EN/DE), where `rating`/`reviews` are largely missing and there is no sponsored flag.
A richer SerpApi re-scrape is used in later iterations (see the repo README).

**Goal.** Check whether Google Shopping's ranking favours certain sellers **beyond** the observable
merit. We train a model to reconstruct Google's order from observable features with
**LightGBM/LambdaMART** grouped by SERP (`run_id`), evaluate with **NDCG@10** on held-out SERPs and
interpret with **SHAP**. We compare two feature sets:
- **merit only**: price, keyword↔title relevance, title length, category, branded/generic;
- **merit + seller**: adds `is_amazon`, `is_giant`, `seller_freq_log`.

If the second reconstructs the order **better**, the seller carries information beyond measurable
merit -> a *candidate* bias signal. SHAP then gives the **direction** (is Amazon pushed up or down?).

> Associative estimate, not causal. Part of the lift can be unmeasured merit (rating, sponsored,
> true relevance) that "hides" in the seller identity. Also recall the data gaps: `rating`/`reviews`
> empty for ~98.6% of rows and **no sponsored flag**.

## 0. GPU test
MiniLM also runs on CPU, but a GPU makes encoding much faster (needs the torch `cu124` build).

In [ ]:
import torch
print("torch:", torch.__version__, "| CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0),
          "| VRAM:", round(torch.cuda.get_device_properties(0).total_memory/1e9,1), "GB")
    y = (torch.rand(5000,5000,device="cuda") @ torch.rand(5000,5000,device="cuda")); torch.cuda.synchronize()
    print("GPU matmul smoke test OK:", tuple(y.shape))
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("selected device:", DEVICE)

## 1. Config
- `ENCODER="sentence-transformers"` uses the **full MiniLM**.
- `ST_LOCAL_PATH`: if set (unzipped `minilm_it/` folder), loads the model **offline** without
  downloading from Hugging Face. Leave empty to download it on first run (~470 MB).

In [ ]:
DB_PATH = "results.db"
COUNTRY = "IT"                       # 'IT' | 'DE' | 'EN'
ENCODER = "sentence-transformers"    # 'sentence-transformers' | 'model2vec' | 'tfidf'
N_SEEDS = 10
ST_LOCAL_PATH = ""                   # e.g. "minilm_it" -> offline; "" -> download from HF

ST_MODEL  = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
M2V_MODEL = "minishlab/potion-multilingual-128M"

import warnings, sqlite3; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import lightgbm as lgb, shap
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import ndcg_score

## 2. Keyword↔title relevance
Three interchangeable encoders. The "real" one is **MiniLM** (semantic): it catches synonyms
(*"running shoes"* ↔ *"trainers"*) that lexical TF-IDF misses. For efficiency we encode only the
**unique** strings (keywords repeat a lot across the ~1.5k SERPs). Automatic TF-IDF fallback if the
encoder is unavailable.

In [ ]:
def _cosine_rows(K, T):
    K = K/(np.linalg.norm(K,axis=1,keepdims=True)+1e-9); T = T/(np.linalg.norm(T,axis=1,keepdims=True)+1e-9)
    return (K*T).sum(1)
def _encode_unique(enc, texts):
    uniq = pd.Index(texts.fillna("").unique()); emb = np.asarray(enc(uniq.tolist()))
    return emb[uniq.get_indexer(texts.fillna(""))]

def rel_sentence_transformers(df):
    from sentence_transformers import SentenceTransformer
    src = ST_LOCAL_PATH or ST_MODEL
    m = SentenceTransformer(src, device=DEVICE, **({"local_files_only": True} if ST_LOCAL_PATH else {}))
    enc = lambda xs: m.encode(xs, normalize_embeddings=True, show_progress_bar=False, batch_size=256)
    return (_encode_unique(enc, df["keyword"]) * _encode_unique(enc, df["title"])).sum(1)

def rel_model2vec(df):
    from model2vec import StaticModel
    m = StaticModel.from_pretrained(M2V_MODEL); enc = lambda xs: m.encode(xs, show_progress_bar=False)
    return _cosine_rows(_encode_unique(enc, df["keyword"]), _encode_unique(enc, df["title"]))

def rel_tfidf(df):
    from sklearn.feature_extraction.text import TfidfVectorizer
    vec = TfidfVectorizer(lowercase=True, ngram_range=(1,2), min_df=2, max_features=20000)
    M = vec.fit_transform(pd.concat([df["keyword"], df["title"].fillna("")])); n=len(df); K,T=M[:n],M[n:]
    kn=np.sqrt(np.asarray(K.multiply(K).sum(1)).ravel()); tn=np.sqrt(np.asarray(T.multiply(T).sum(1)).ravel())
    return np.asarray(K.multiply(T).sum(1)).ravel()/np.where(kn*tn==0,1,kn*tn)

def compute_relevance(df, encoder):
    fn={"sentence-transformers":rel_sentence_transformers,"model2vec":rel_model2vec,"tfidf":rel_tfidf}[encoder]
    try: return fn(df), encoder
    except Exception as e:
        print(f"[rel] '{encoder}' unavailable ({type(e).__name__}: {e}) -> TF-IDF fallback")
        return rel_tfidf(df), "tfidf(fallback)"

## 3. Data and features
Load the chosen market, keep valid prices only, compute relevance and build the features.
The label `y` is graded from the position (top-2 -> 4, ..., beyond 20th -> 0).

In [ ]:
con=sqlite3.connect(DB_PATH); df=pd.read_sql("SELECT * FROM products WHERE language=?",con,params=(COUNTRY,)); con.close()
df=df[df["price_value"].notna() & (df["price_value"]>0)].reset_index(drop=True)
print(f"rows {len(df):,} | SERPs {df['run_id'].nunique()}")

rel, used = compute_relevance(df, ENCODER); df["rel"]=rel
print(f"relevance: {used} | mean={rel.mean():.3f} std={rel.std():.3f}")

t=df["title"].fillna("")
df["log_price"]=np.log1p(df["price_value"]); df["title_len"]=t.str.len(); df["title_words"]=t.str.split().apply(len)
df["is_branded"]=(df["query_type"]=="branded").astype(int)
df["is_amazon"]=df["seller"].fillna("").str.contains("amazon",case=False).astype(int)
df["is_giant"]=df["seller"].isin(df["seller"].value_counts().head(15).index).astype(int)
df["seller_freq_log"]=np.log1p(df["seller"].map(df["seller"].value_counts()).fillna(0))
for c in ("category_l1","category_l2"): df[c]=df[c].astype("category")
df["y"]=df["position"].apply(lambda p:4 if p<=2 else 3 if p<=5 else 2 if p<=10 else 1 if p<=20 else 0)

MERIT=["log_price","rel","title_len","title_words","is_branded","category_l1","category_l2"]
PLATFORM=["is_amazon","is_giant","seller_freq_log"]; CAT=["category_l1","category_l2"]

## 4. Baselines and models (single split)
Two reference baselines — **random** order and **price-ascending** order — plus the two models.
Everything evaluated on the **same test SERPs** to stay comparable.

In [ ]:
def split(df, seed=42):
    return next(GroupShuffleSplit(1,test_size=0.3,random_state=seed).split(df,groups=df["run_id"]))
def grp(d): return d.groupby("run_id",sort=False).size().values

def ndcg_baseline(dte, score):
    tot=0.0; k=0
    for _,g in dte.groupby("run_id",sort=False):
        if len(g)<2: continue
        tot += ndcg_score([g["y"].values],[score(g)],k=10); k+=1
    return tot/k

def train(df, feats, tr, te):
    dtr=df.iloc[tr].sort_values("run_id"); dte=df.iloc[te].sort_values("run_id")
    m=lgb.LGBMRanker(objective="lambdarank",metric="ndcg",n_estimators=300,learning_rate=0.05,num_leaves=31,
        min_child_samples=30,subsample=0.8,colsample_bytree=0.8,random_state=0,n_jobs=2,verbose=-1,label_gain=[0,1,3,7,15])
    m.fit(dtr[feats],dtr["y"],group=grp(dtr),eval_set=[(dte[feats],dte["y"])],eval_group=[grp(dte)],
          eval_at=[10],categorical_feature=[c for c in CAT if c in feats])
    return m, dte, m.best_score_["valid_0"]["ndcg@10"]

tr,te=split(df,42); dte_all=df.iloc[te].sort_values("run_id")
rng=np.random.default_rng(0)
nd_rand  = ndcg_baseline(dte_all, lambda g: rng.random(len(g)))
nd_price = ndcg_baseline(dte_all, lambda g: -g["price_value"].values)   # price ascending
mA,_,nA  = train(df, MERIT, tr, te)
mB,dte,nB= train(df, MERIT+PLATFORM, tr, te)
results={"random":nd_rand,"price ascending":nd_price,"merit only":nA,"merit + seller":nB}
for k,v in results.items(): print(f"  {k:<18} NDCG@10={v:.3f}")
print(f"\nSELLER LIFT: {nB-nA:+.3f}")

### Figure 1 — NDCG@10 by model
How well each strategy reconstructs Google's order. The jump from *merit only* to *merit+seller*
is the **lift**.

In [ ]:
fig,ax=plt.subplots(figsize=(7,3.6))
ks=list(results); vs=[results[k] for k in ks]
cols=["#bdbdbd","#bdbdbd","#4c78a8","#e45756"]
b=ax.barh(ks, vs, color=cols); ax.invert_yaxis(); ax.set_xlim(0.35, max(vs)+0.03)
for r,v in zip(b,vs): ax.text(v+0.003, r.get_y()+r.get_height()/2, f"{v:.3f}", va="center", fontsize=10)
ax.set_xlabel("NDCG@10 (held-out SERPs)"); ax.set_title(f"Reconstructing Google's order — {COUNTRY} ({used})")
ax.annotate(f"lift {nB-nA:+.3f}", xy=(nB,3), xytext=(nA,3.35),
            arrowprops=dict(arrowstyle="->"), fontsize=9, color="#e45756")
plt.tight_layout(); plt.savefig("nb_ndcg.png",dpi=120,bbox_inches="tight"); plt.show()

## 5. Stability across splits
A single SERP split is noisy: repeat over `N_SEEDS` different partitions and look at the
mean ± std of the lift and of the `is_amazon` SHAP value.

In [ ]:
rows=[]
for s in range(N_SEEDS):
    tr,te=split(df,s)
    _,_,a=train(df,MERIT,tr,te); mB,dte_s,b=train(df,MERIT+PLATFORM,tr,te)
    sv=shap.TreeExplainer(mB).shap_values(dte_s[MERIT+PLATFORM])
    ia=dte_s["is_amazon"].values; sa=sv[:,(MERIT+PLATFORM).index("is_amazon")][ia==1].mean()
    rows.append((a,b,b-a,sa))
A=np.array(rows)
print(f"encoder={used} | {N_SEEDS} seeds")
print(f"  merit only : {A[:,0].mean():.3f} ± {A[:,0].std():.3f}")
print(f"  +seller    : {A[:,1].mean():.3f} ± {A[:,1].std():.3f}")
print(f"  lift       : {A[:,2].mean():+.3f} ± {A[:,2].std():.3f}")
print(f"  is_amazon  : {A[:,3].mean():+.3f} ± {A[:,3].std():.3f}")

### Figure 2 — Lift distribution across seeds
If the cloud of points sits entirely above zero, the lift is **robust** (the seller always helps).

In [ ]:
fig,ax=plt.subplots(figsize=(7,3.2))
ax.axvline(0,color="#999",lw=1,ls="--")
ax.scatter(A[:,2], np.zeros(len(A))+rng.normal(0,0.03,len(A)), color="#e45756", alpha=0.8, zorder=3)
ax.errorbar(A[:,2].mean(),0, xerr=A[:,2].std(), fmt="o", color="black", capsize=4, zorder=4,
            label=f"mean {A[:,2].mean():+.3f} ± {A[:,2].std():.3f}")
ax.set_yticks([]); ax.set_xlabel("lift NDCG@10 (merit+seller − merit only)")
ax.set_title("Seller lift across {} splits".format(N_SEEDS)); ax.legend(loc="upper left", fontsize=9)
plt.tight_layout(); plt.savefig("nb_lift.png",dpi=120,bbox_inches="tight"); plt.show()

### Figure 3 — SHAP for the merit+seller model
Direction and strength of each feature. Look at `is_amazon`: if the `amazon=1` points sit at
**negative** SHAP, Amazon is pushed **down** (no favouritism). `seller_freq_log` (prevalence) is
usually the strongest platform feature.

In [ ]:
sv=shap.TreeExplainer(mB).shap_values(dte[MERIT+PLATFORM])
shap.summary_plot(sv, dte[MERIT+PLATFORM], show=False, max_display=10)
plt.tight_layout(); plt.savefig("nb_shap.png",dpi=120,bbox_inches="tight"); plt.show()

## 6. Summary table → CSV

In [ ]:
summary=pd.DataFrame({
 "metric":["NDCG random","NDCG price","NDCG merit only","NDCG +seller",
           "lift (mean)","lift (std)","is_amazon SHAP (mean)","is_amazon SHAP (std)"],
 "value":[nd_rand,nd_price,A[:,0].mean(),A[:,1].mean(),A[:,2].mean(),A[:,2].std(),A[:,3].mean(),A[:,3].std()],
 "country":COUNTRY,"encoder":used,"n_seeds":N_SEEDS})
summary.to_csv(f"bias_summary_{COUNTRY}_{used}.csv", index=False)
print(summary.to_string(index=False))

## How to read the results
- **Lift > 0 and stable** -> the seller carries information beyond merit -> *candidate* bias (not causal).
- **`is_amazon` SHAP negative** -> no pro-Amazon favouritism (if anything, penalised).
- **Encoder comparison**: the hypothesis was that *true semantic* relevance would raise merit-only and
  **reduce** the lift (absorbing relevance that hid in the seller identity). Re-run with
  `ENCODER="tfidf"` and compare: if the lift does **not** drop, it is driven by the seller's
  **prevalence** (`seller_freq_log`), not by unmeasured relevance.

**Structural limits:** without `rating` and without a **sponsored flag**, the quality-adjusted test
and a direct measure of paid-placement bias remain impossible on this reduced dataset.